In [85]:
%reset -f

In [86]:
from get_data import get_processed_path
import warnings
import logging

import pandas as pd
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
import pickle
import os


import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report

import re
from sklearn.preprocessing import RobustScaler
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras import callbacks
import pickle
import numpy as np

warnings_logger = logging.getLogger('warnings')
warnings_logger.setLevel(logging.WARNING)
warning_handler = logging.FileHandler('warnings.log')
warning_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
warnings_logger.addHandler(warning_handler)

def warning_handler_func(message, category, filename, lineno, file=None, line=None):
    warnings_logger.warning(f"{category.__name__}: {message} (File: {filename}, Line: {lineno})")

warnings.showwarning = warning_handler_func

In [87]:
processed_forecasting_path, processed_statuses_path = get_processed_path()

df_removed_nans_forecasting = pd.read_csv(processed_forecasting_path, index_col=0, parse_dates=True)
df_removed_nans_classification = pd.read_csv(processed_statuses_path, index_col=0, parse_dates=True)
# to do in future: train model on input data from different dates
# merge data
df = pd.merge(df_removed_nans_forecasting, df_removed_nans_classification, on=['timestamp'])

In [88]:
def extract_port_numbers(string_input):
    """Extract all numbers from interface string, handling multi-digit ports"""
    numbers = re.findall(r'\d+', string_input)
    return ''.join(numbers) if numbers else None

def encode_column_names(df):
    encoded_columns = {}
    unique_ports = []

    for column in df.columns:
        if "interface" in column.lower() and "/" in column:
            port_strings = column.split("/")
            port_numbers = []
            for port_string in port_strings:
                #num = extract_int(port_string)
                num = extract_port_numbers(port_string)
                if num is not None:
                    port_numbers.append(str(num))
            encoded_columns[column] = "".join(port_numbers)
        elif ": operational status" in column.lower() and "/" in column:
            encoded_columns[column] = column

    for idx, name in encoded_columns.items():
        if ": operational status" in idx.lower():
            map_name = f"Status {name}"
            encoded_columns[idx] = map_name
        elif "bits sent" in idx.lower():
            map_name = f"Bits Sent {name}"
            encoded_columns[idx] = map_name
        elif "bits received" in idx.lower():
            map_name = f"Bits Received {name}"
            encoded_columns[idx] = map_name

        if int(name) not in unique_ports:
            unique_ports.append(int(name))
    
    return encoded_columns, sorted(unique_ports)

encoded_columns, unique_ports = encode_column_names(df)
df.rename(columns=encoded_columns, inplace=True)

In [89]:
def encode_traffic_ratio_labels_per_port(
    df,
    threshold,
    unique_ports,
    start_bit=10,
    epsilon=1e-6
):
    """
    (docstring omitted for brevity — see previous assistant message)
    """
    import pandas as pd
    import numpy as np

    unique_ports = list(unique_ports)
    df_encoded = df.copy()

    # Precompute ratios per port and record which ports are valid
    traffic_ratio_dict = {}
    valid_ports = []
    for port in unique_ports:
        sent_col = f"Bits Sent {port}"
        recv_col = f"Bits Received {port}"
        if sent_col in df.columns and recv_col in df.columns:
            sent = pd.to_numeric(df[sent_col], errors='coerce').fillna(0).astype(float)
            recv = pd.to_numeric(df[recv_col], errors='coerce').fillna(0).astype(float)
            ratio = recv / (sent + float(epsilon))
            traffic_ratio_dict[port] = ratio
            valid_ports.append(port)
        else:
            traffic_ratio_dict[port] = None
            print(f"Warning: Columns for port {port} not found in dataframe")

    contributions = []
    for i, port in enumerate(unique_ports):
        ratio = traffic_ratio_dict.get(port)
        bit_pos = start_bit + i
        if ratio is None:
            contributions.append(pd.Series(0, index=df.index, dtype=object))
            continue
        mask = (ratio > threshold)
        contribution = mask.astype(object) * (1 << bit_pos)
        contributions.append(contribution)
        print(f"Port {port} (bit {bit_pos}) - Ratio > {threshold}: {mask.sum()}")

    if contributions:
        total = pd.Series(0, index=df.index, dtype=object)
        for c in contributions:
            total = total + c
        df_encoded['TrafficRatioLabel'] = total
    else:
        df_encoded['TrafficRatioLabel'] = 0

    print(f"Traffic ratio encoding summary:")
    print(f"  - Threshold: {threshold}")
    print(f"  - Unique ports requested: {len(unique_ports)} (valid: {len(valid_ports)})")
    print(f"  - Bit range: {start_bit} to {start_bit + len(unique_ports) - 1}")
    for i, port in enumerate(unique_ports):
        bit_pos = start_bit + i
        has_data = traffic_ratio_dict.get(port) is not None
        print(f"  - Port {port}: bit {bit_pos} {'✓' if has_data else '✗ (missing data)'}")
    if start_bit + len(unique_ports) > 63:
        print("Warning: total bit range exceeds 64 bits — be careful if converting to fixed-size integers (np.int64).")

    for idx, value in traffic_ratio_dict.items():
        print(f"{idx}: value={value}")

    return df_encoded


In [94]:
df_labeled = encode_traffic_ratio_labels_per_port(df, threshold=100, unique_ports=unique_ports)

Port 101 (bit 10) - Ratio > 100: 0
Port 103 (bit 12) - Ratio > 100: 0
Port 111 (bit 17) - Ratio > 100: 0
Port 114 (bit 18) - Ratio > 100: 10871
Port 201 (bit 19) - Ratio > 100: 8
Port 205 (bit 23) - Ratio > 100: 0
Port 211 (bit 25) - Ratio > 100: 4
Port 212 (bit 26) - Ratio > 100: 1
Port 214 (bit 27) - Ratio > 100: 20502
Traffic ratio encoding summary:
  - Threshold: 100
  - Unique ports requested: 18 (valid: 9)
  - Bit range: 10 to 27
  - Port 101: bit 10 ✓
  - Port 102: bit 11 ✗ (missing data)
  - Port 103: bit 12 ✓
  - Port 104: bit 13 ✗ (missing data)
  - Port 107: bit 14 ✗ (missing data)
  - Port 108: bit 15 ✗ (missing data)
  - Port 109: bit 16 ✗ (missing data)
  - Port 111: bit 17 ✓
  - Port 114: bit 18 ✓
  - Port 201: bit 19 ✓
  - Port 202: bit 20 ✗ (missing data)
  - Port 203: bit 21 ✗ (missing data)
  - Port 204: bit 22 ✗ (missing data)
  - Port 205: bit 23 ✓
  - Port 208: bit 24 ✗ (missing data)
  - Port 211: bit 25 ✓
  - Port 212: bit 26 ✓
  - Port 214: bit 27 ✓
101: value=

In [91]:
label = df_labeled['TrafficRatioLabel']

In [96]:
import pandas as pd

def decode_traffic_ratio_label(label, unique_ports, start_bit=10):
    ports = list(unique_ports)
    bit_positions = [start_bit + i for i in range(len(ports))]

    def decode_single(val):
        return {port: bool((int(val) >> bit) & 1) for port, bit in zip(ports, bit_positions)}

    if hasattr(label, "index"):  # pandas Series
        decoded = [decode_single(val) for val in label]
        return pd.DataFrame(decoded, index=label.index)
    else:
        return decode_single(label)

In [98]:
label_to_name = {}

for label in df_labeled['TrafficRatioLabel'].unique()[:5]:
    decoded = decode_traffic_ratio_label(label, unique_ports)
    decoded = {k: v for k, v in decoded.items() if v}  
    label_to_name[label] = decoded
    print(f"Label: {label} -> Decoded: {decoded}")

Label: 262144 -> Decoded: {114: True}
Label: 0 -> Decoded: {}
Label: 134217728 -> Decoded: {214: True}
Label: 134479872 -> Decoded: {114: True, 214: True}
Label: 67371008 -> Decoded: {114: True, 212: True}
